In [1]:
import lsdb
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nested_pandas as npd
from nested_pandas.utils import count_nested
from importlib import reload
import multiband_fit_template_numba as mbft
from scipy.signal import find_peaks
# from astropy.timeseries import LombScargle, LombScargleMultiband # optional — for comparing

/astro/users/scharney/.conda/envs/lsdb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We're gonna start with a gigantic dataset and whittle it down to a more managable size. Then, run RR Lyrae template fitting code on what's left to get a bunch of parameters to try to tell them apart

In [2]:
objcat = lsdb.open_catalog('/epyc/data3/hats/catalogs/dp1/object_collection', columns=['objectId', 'coord_dec', 'coord_ra', 'objectForcedSource', 'u_psfMag', 'u_psfMagErr',
                                                                                   'g_psfMag', 'g_psfMagErr', 'r_psfMag', 'r_psfMagErr', 'i_psfMag', 'i_psfMagErr', 'z_psfMag', 'z_psfMagErr',
                                                                                   'y_psfMag', 'y_psfMagErr', 'ebv']) # DP1 Object catalog with only the columns I need
objall = lsdb.open_catalog('/epyc/data3/hats/catalogs/dp1/object_collection', columns='all')

In [3]:
# load in the fitter template
tem = mbft.load_template_dir("lsst_template")

In [4]:
# a bunch of helper functions to help us whittle down the big data set
# this one does what it says on the tin
def count_points(df, new_name='n_lc'):
    # Asked to count `lc`, this will add a column called `n_lc`
    return count_nested(df, "objectForcedSource").rename(columns={'n_objectForcedSource':new_name})

def calc_std_partition(df, band='g'):
    df = npd.NestedFrame(df)
    
    # filter to requested band (flags already removed by filter_flags_partition)
    lc = df.query(f"objectForcedSource.band == '{band}'")['objectForcedSource']
    
    if len(lc) == 0:
        return df.assign(std=np.nan)
    
    # vectorized min/max over all objects at once
    std = lc['psfMag'].groupby(level=0).std()
    
    # objects with no observations in this band get nan automatically
    std.name = 'std'
    return df.join(std)

def dust_correction_single_band(df, band):
    A_band = df['ebv'] * tem['dust'][band]
    corrected = (df[f'{band}_psfMag'] - A_band)
    df[f'{band}_psfMagExt'] = corrected
    return df

def dust_correction(df):    
    for band in tem['dust'].keys():
        df = dust_correction_single_band(df, band)
    return df


In [9]:
%%time
cat = objcat # pick your favorite catalog
print(f"Starting length: {len(cat)} objects")
sample = cat.head(1)['objectForcedSource'].iloc[0] # pull out a single light curve
FLAG_COLS = [col for col in sample.columns if 'flag' in col.lower()] #do this once, so we don't have to calc it again

# correct our observations for the effects of dust (minor)
dust_corrected = cat.map_partitions(dust_correction)

# a bunch of color cuts to roughly limit the space of objects we expect to RR Lyrae
# this isn't the best, especially early on, but it's good enough
ug_query = 'u_psfMagExt - g_psfMagExt > 0.500 and u_psfMagExt - g_psfMagExt < 1.4'
gr_query = 'g_psfMagExt - r_psfMagExt > -0.15 and g_psfMagExt - r_psfMagExt < 0.4'
ri_query = 'r_psfMagExt - i_psfMagExt > -0.25 and r_psfMagExt - i_psfMagExt < 0.3'
iz_query = 'i_psfMagExt - z_psfMagExt > -0.21 and i_psfMagExt - z_psfMagExt < 0.45'
zy_query = 'z_psfMagExt - y_psfMagExt > -0.22 and z_psfMagExt - y_psfMagExt < 0.15'

# actually apply the color cuts
color_query = f'{ug_query} and {gr_query} and {ri_query} and {iz_query} and {zy_query}'
color_cut = dust_corrected.query(color_query)
# remove all observations that are flagged
flag_query = " and ".join(f"objectForcedSource.{col} == False" for col in FLAG_COLS)
flag_cut = color_cut.query(flag_query)

# count how many points are in each light curve
counted = flag_cut.map_partitions(count_points)
num_cut = counted.query("n_lc >= 20") # tunable
# simple variability cut
# make sure the object is changing by enforcing some non-negligible STD cut
# a flat object will have a super low std
with_std = num_cut.map_partitions(calc_std_partition)
std_cut = with_std.query('std >= 0.1')

cdf = std_cut.compute() # cut data frame
print(f"Final length: {len(cdf)} objects")

Starting length: 2299726 objects


Computing Catalog: 100%|██████████| 5058/5058 [01:00<00:00, 83.52it/s] 

Final length: 3551 objects
CPU times: user 2min 19s, sys: 31.4 s, total: 2min 50s
Wall time: 1min 3s


In [12]:
filter_colors = {'u': '#0c71ff', 'g': '#49be61', 'r': '#c61c00', 'i': '#ffc200', 'z': '#f341a2', 'y': '#5d0000'}

def template_fitting(tem, lc, print_outputs = False, use_dust = False, multiband = True, fit_n = 20, coeff_n = 10, omega_n = 100.0, period_range=[0.2, 0.9], cols=['midpointMjdTai', 'band', 'psfMag', 'psfMagErr']):
    '''
    Returns a dictionary with coeffs (mu, d, a, phi), pests (top 3), cov (of linear params: mu (distance modulus), d (dust), a (amplitude)), sigma_P (local uncertainty of a given period), 
        like_P (global posterior likelihood uncertainty), the next best 3 periods, gen_lc (the generated light curve)
    '''
    # compute best coefficients
    omegas = np.arange(1/period_range[-1], 1/period_range[0], 0.1/omega_n) #periods from [0.2, 0.9]:[1.1, 5.0], now trying [0.3, 0.8]

    lc_bands    = list(lc[cols[1]].unique())
    # print(lc_bands)
    if multiband:
        rss = mbft.FitTemplate_multiband(tem, lc, omegas, NN=fit_n, use_errors=True, use_dust=False, use_band_shift=True, cols = cols)
    else:
        rss = mbft.FitTemplate_multiband(tem, lc, omegas, NN=fit_n, use_errors=True, use_dust=False, use_band_shift=False, cols = cols)

    rss = np.array(rss)
    
    best_omega = omegas[np.argmin(rss)]
    best_pest = 1/best_omega
    
    coeffs, cov = mbft.ComputeCoeffsAndCov_multiband(tem, lc, float(best_omega), NN=coeff_n, use_errors=True, use_dust=False, cols = cols)

    # calculate error and posterior on period
    chi2_min = np.min(rss) # rss is chi2 bc it's already scaled by weights

    chi2_red = chi2_min / (len(lc.midpointMjdTai) - len(coeffs)) # length - dof
        
    periods = 1/np.array(omegas)
    rms = np.sqrt(rss / (len(lc.midpointMjdTai) - len(coeffs))) # length - dof
    mask = rss <= chi2_min + 2.3
   
    sigma_P = 0.5 * np.abs(periods[mask][0] - periods[mask][-1])

    # now get posterior likelihood to figure out how likely this is to be the right answer
    peaks, props = find_peaks(-rss, prominence=10, width=0.1)
    try:
        next_best_idx = np.argpartition(props['prominences'], -3)[-3:]
        next_best = periods[peaks[next_best_idx]]
        next_best_chi = rss[peaks[next_best_idx]]
    except:
        next_best = []

    L = np.exp(-0.5 * (rss - rss.min()))  # subtract min to prevent underflow
    norm = np.trapezoid(L, periods) 
    if norm == 0 or ~np.isfinite(norm):
        like_P = np.nan
    else:
        L /= norm # prob density
        P_mean = np.trapezoid(periods * L, periods) # mean of posterior
        P_var  = np.trapezoid((periods - P_mean)**2 * L, periods) # variance on posterior
        like_P = np.sqrt(P_var) # std/likelihood of posterior

    if print_outputs:
        print("omega_best:", best_omega)
        print(f"pest: {best_pest:0.4f} +- {sigma_P:0.6f}, global uncertainty: {like_P:0.4f}")
        print("coeffs (mu, d, a, phi):", coeffs)  # [mu, d, a, phi]
        # print("cov (mu, d, a):\n", cov)

    return dict({'coeffs':coeffs, 'p_est':best_pest, 'cov':cov, 'variance':sigma_P, 'posterior':like_P, 'next_best':next_best, 'rss':rss})

def fit_stat_df(df, nested_column='objectForcedSource'):
    cdf = df.copy() # for changing df

    # initialize all the new columns so it doesn't yell at me
    for col in ["fit_coeffs", "flat_chi2_red", "chi2_red", "wrms", "p_est", "p_err", "wrms_ratio"]:
        cdf[col] = np.nan

    # these ones are gonna be dictionaries (there's probably a better data structure for this)
    cdf["fit_coeffs"] = pd.Series(dtype=object)
    cdf["flat_chi2_red"] = pd.Series(dtype=object)
    cdf["chi2_red"] = pd.Series(dtype=object)

    # find a template fit to each object in the table, save the results
    for idx, row in cdf.iterrows(): 
        lc_raw = row[nested_column].dropna(subset=['psfMag', 'psfMagErr', 'midpointMjdTai'])
        flag_cols = [col for col in lc_raw.columns if 'flag' in col.lower()]
        lc_flag = lc_raw[~lc_raw[flag_cols].any(axis=1)]
    
        cols=['midpointMjdTai', 'band', 'psfMag', 'psfMagErr']
        band_counts = lc_flag.groupby(cols[1])[cols[2]].count()
        valid_bands = band_counts[band_counts >= 3].index.tolist()
        
        # filter lc_sorted to only valid bands
        lc = lc_flag[lc_flag[cols[1]].isin(valid_bands)].reset_index(drop=True)
        lc_bands = list(lc[cols[1]].unique())
        if len(lc_bands) <= 1:
            print(f"{row.name} doesn't have enough bands for fitting. Skipping...")
            continue
        if len(lc) < 10:
            print(f"{row.name} only has {len(lc)} points and is unsuitable for fitting. Skipping...")
            continue
        
        pipeline_output = template_fitting(tem, lc, omega_n=20, print_outputs=False, cols=cols)
        
        coeffs = pipeline_output['coeffs']
        pest = pipeline_output['p_est']
        tem_plot = mbft.reorder_template_for_lc(tem, lc_bands)
        gamma = tem_plot['templates']
        t = tem_plot['temp_time']
        
        abs_mag_est = tem_plot['abs_mag'](pest, tem_plot)[0]
        chi2_total = 0
        flat_chi2_total = 0
        chi2_dict = dict({})
        flat_chi2_dict = dict({})
        wrms_band = []
        scatter_ratios = []
        coeffs_out = dict({'mu': coeffs[0], 'd': coeffs[1], 'a':coeffs[2], 'phi':coeffs[3]})
        for i, band in enumerate(lc_bands):
            
            gamma_band = gamma[i, :]
            
            offset = coeffs[4 + i] if i < len(lc_bands)-1 else 0.0
            coeffs_out[band] = offset
            
            m_est=coeffs[0] + abs_mag_est[i] + coeffs[2]*gamma_band + offset
    
            band_lc = lc[lc['band'] == band]
            
            minterp = np.interp(band_lc.midpointMjdTai%pest/pest, t, m_est)
        
            model_err = tem['model_error']['g'] # same for all bands
    
            err_total = np.sqrt(band_lc.psfMagErr**2 + model_err**2)
    
            chi2 = np.sum(((band_lc.psfMag - minterp) ** 2) / err_total**2)
            chi2_total += chi2
    
            chi2_dict[band] = chi2
    
            weights = 1/err_total**2
    
            flat = np.average(band_lc.psfMag, weights=weights)
            flat_chi2 = np.sum(weights * (band_lc.psfMag - flat)**2)
    
            flat_chi2_dict[band] = flat_chi2
            flat_chi2_total += flat_chi2
    
            wrms_template = np.sqrt(np.average((band_lc.psfMag - minterp) ** 2, weights=weights))
            wrms_band.append(wrms_template)
            wrms_flat = np.sqrt(np.average((band_lc.psfMag - flat)**2, weights=weights))
            
            # Ratio: < 1 means template is doing something useful
            scatter_ratio = wrms_template / wrms_flat
            scatter_ratios.append(scatter_ratio)
            
        dof = len(lc.psfMag) - len(coeffs)
        flat_dof = len(lc.psfMag) - len(lc_bands)
        chi2_red = chi2_total/dof
        flat_chi2_red = flat_chi2_total/flat_dof
    
        
        # Weighted by number of points per band
        band_sizes = [len(lc[lc['band']==b]) for b in lc_bands]
        scatter_ratio_global = np.average(scatter_ratios, weights=band_sizes)
    
        wrms = np.average(np.array(wrms_band), weights=band_sizes)
    
        chi2_dict['total'] = chi2_red
        flat_chi2_dict['total'] = flat_chi2_red
        
        cdf.at[row.name, "chi2_red"] = chi2_dict
        cdf.at[row.name, "flat_chi2_red"] = flat_chi2_dict
        cdf.loc[row.name, "wrms"] = wrms
        cdf.loc[row.name, "p_est"] = pest
        cdf.loc[row.name, "p_err"] = pipeline_output['posterior']
        cdf.loc[row.name, "wrms_ratio"] = scatter_ratio_global
        cdf.at[row.name, "fit_coeffs"] = coeffs_out
    return cdf

In [13]:
%%time
sdf = fit_stat_df(cdf) #statistic data frame
sdf

CPU times: user 31min 13s, sys: 770 ms, total: 31min 14s
Wall time: 22min 27s


objectId  coord_dec   coord_ra  \
_healpix_29                                                     
1450079808718075624  612929216534556056 -25.485147  95.086896   
1450085790055887358  612929216534557319 -25.440082  95.087111   
...                                 ...        ...        ...   
2530258924158150677  611256653190273049 -27.655111  52.683631   
2530266718820929954  611256515751334023 -27.567939  53.063874   

                                                    objectForcedSource  \
_healpix_29                                                              
1450079808718075624  [{parentObjectId: 0, coord_ra: 95.086896, coor...   
1450085790055887358  [{parentObjectId: 0, coord_ra: 95.087111, coor...   
...                                                                ...   
2530258924158150677  [{parentObjectId: 0, coord_ra: 52.683631, coor...   
2530266718820929954  [{parentObjectId: 0, coord_ra: 53.063874, coor...   

                      u_psfMag  u_psfMagErr   g_psfMag  g_psfMagErr  \
_healpix_29                                                           
1450079808718075624  24.604836     0.423083  23.978354     0.044029   
1450085790055887358  24.655766     0.336385  23.623724     0.022122   
...                        ...          ...        ...          ...   
2530258924158150677  24.457458     0.560378  23.808451     0.036128   
2530266718820929954  25.401951     1.880401  24.420017      0.04025   

                      r_psfMag  r_psfMagErr  ...  y_psfMagExt  n_lc       std  \
_healpix_29                                  ...                                
1450079808718075624  23.676901     0.047515  ...    23.312832    36  0.234981   
1450085790055887358  23.249641     0.021555  ...    22.713658    46  0.264485   
...                        ...          ...  ...          ...   ...       ...   
2530258924158150677  23.413715     0.042525  ...    22.886672    35  0.328189   
2530266718820929954  24.318541     0.083809  ...    24.072645    76  0.315855   

                                                            fit_coeffs  \
_healpix_29                                                              
1450079808718075624  {'mu': 22.398300118198197, 'd': 0.0, 'a': 1.01...   
1450085790055887358  {'mu': 21.936819990878302, 'd': 0.0, 'a': 0.62...   
...                                                                ...   
2530258924158150677  {'mu': 22.118637576966027, 'd': 0.0, 'a': 0.73...   
2530266718820929954  {'mu': 22.76869134992501, 'd': 0.0, 'a': 0.284...   

                                                         flat_chi2_red  \
_healpix_29                                                              
1450079808718075624  {'r': 49.047188, 'g': 17.863808, 'z': 7.131301...   
1450085790055887358  {'i': 2.1865647, 'r': 60.018696, 'g': 61.9703,...   
...                                                                ...   
2530258924158150677  {'i': 8.925503, 'r': 6.7176857, 'g': 13.564861...   
2530266718820929954  {'g': 25.36912, 'i': 9.279602, 'r': 2.4412346,...   

                                                              chi2_red  \
_healpix_29                                                              
1450079808718075624  {'r': 219.76492793252862, 'g': 134.05459686232...   
1450085790055887358  {'i': 82.39964128582089, 'r': 91.3335135255654...   
...                                                                ...   
2530258924158150677  {'i': 81.85258979430762, 'r': 73.7009914635221...   
2530266718820929954  {'g': 33.041939865756824, 'i': 9.9810223523592...   

                         wrms     p_est     p_err  wrms_ratio  
_healpix_29                                                    
1450079808718075624  0.514719  0.207852  0.084356    1.974329  
1450085790055887358  0.346219  0.415608  0.077828    1.695424  
...                       ...       ...       ...         ...  
2530258924158150677  0.460028  0.212039  0.194816    3.033602  
2530266718820929954  0.278795  0.312880  0.206071